# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Running in Colab')
    !pip install -q polars

# High-Performance Data Analysis with Polars

Polars is a lightning-fast DataFrame library implemented in Rust and memory-efficient. This notebook demonstrates its syntax and performance advantages over Pandas for large datasets.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Polars version: {pl.__version__}")

## 1. Generate Large Synthetic Dataset

We generate a dataset with 1 million rows to compare performance.

In [ ]:
N = 1_000_000

data = {
    'id': np.arange(N),
    'group': np.random.choice(['A', 'B', 'C', 'D', 'E'], N),
    'value1': np.random.rand(N),
    'value2': np.random.randn(N)
}

# Create Pandas DataFrame
start = time.time()
df_pd = pd.DataFrame(data)
print(f"Pandas creation time: {time.time() - start:.4f}s")

# Create Polars DataFrame
start = time.time()
df_pl = pl.DataFrame(data)
print(f"Polars creation time: {time.time() - start:.4f}s")

## 2. Syntax Comparison: Filter & Select

Polars uses expressions (`pl.col()`) which are very readable and allow for optimization.

In [ ]:
# Pandas: Filter where group is 'A' and select 'value1'
start = time.time()
res_pd = df_pd[df_pd['group'] == 'A']['value1']
pd_time = time.time() - start
print(f"Pandas Filter+Select: {pd_time:.4f}s")

# Polars
start = time.time()
res_pl = df_pl.filter(pl.col('group') == 'A').select(pl.col('value1'))
pl_time = time.time() - start
print(f"Polars Filter+Select: {pl_time:.4f}s")

print(f"Speedup: {pd_time / pl_time:.2f}x")

## 3. GroupBy Aggregations

Polars is exceptionally fast at aggregations.

In [ ]:
# Pandas GroupBy Mean
start = time.time()
agg_pd = df_pd.groupby('group')[['value1', 'value2']].mean()
pd_time = time.time() - start
print(f"Pandas GroupBy: {pd_time:.4f}s")

# Polars GroupBy Mean
start = time.time()
agg_pl = df_pl.group_by('group').agg([
    pl.col('value1').mean(),
    pl.col('value2').mean()
])
pl_time = time.time() - start
print(f"Polars GroupBy: {pl_time:.4f}s")

print(f"Speedup: {pd_time / pl_time:.2f}x")

## 4. Lazy Execution

Polars supports LazyFrames, which build a query plan and only execute when `collect()` is called. This allows the query optimizer to combine operations.

In [ ]:
lazy_query = (
    df_pl.lazy()
    .filter(pl.col('value1') > 0.5)
    .group_by('group')
    .agg(pl.col('value2').sum().alias('total_value2'))
    .sort('total_value2', descending=True)
)

print("Query Plan:")
print(lazy_query.explain())

start = time.time()
result = lazy_query.collect()
print(f"\nLazy Execution Time: {time.time() - start:.4f}s")
print(result)